# E3 - Datos para el modelo, evaluación y arquitectura

El objetivo del proyecto es describir los hábitos registrados y su asociación con la frecuencia cardíaca en reposo. Como análisis complementario, construimos un modelo predictivo exploratorio y evaluamos su desempeño con una separación temporal.

In [1]:
from pathlib import Path
import sys, json, csv
from IPython.display import display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'modelamiento_e3.py').exists()), None)
assert ROOT is not None, 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(ROOT / 'src'))
from modelamiento_e3 import run_model
ROOT

WindowsPath('.')

## 1. Construcción y evaluación temporal

Usamos como referencia una predicción constante igual a la mediana de los valores del periodo de entrenamiento. La comparamos con una regresión lineal estandarizada. Entrenamos ambos modelos con fechas hasta el corte temporal y evaluamos sus resultados con las fechas posteriores.

In [2]:
metricas_e3 = run_model(ROOT)
print(json.dumps(metricas_e3, ensure_ascii=False, indent=2))

{
  "objetivo": "frecuencia cardíaca en reposo del día siguiente (bpm)",
  "ventana_features": "7 días anteriores; no incluye frecuencia cardíaca como predictor",
  "particion": "temporal global: train hasta 2008-02-02; test posterior. Los usuarios pueden aparecer en ambos periodos; ninguna variable posterior a fecha_features_hasta se usa como predictor",
  "fecha_corte_temporal": "2008-02-02",
  "features": [
    "pasos_7d_promedio",
    "minutos_sesion_7d_promedio",
    "comidas_7d",
    "porciones_7d",
    "calorias_7d",
    "sodio_7d",
    "grasa_saturada_7d",
    "azucar_7d",
    "fibra_7d",
    "proteina_7d"
  ],
  "nutrition_feature_policy": "7-day sums over matched, logged meals; no-meal days contribute zero recorded intake; windows with meal events missing nutrient joins are excluded",
  "n_rows_total": 10710,
  "n_rows_train": 8445,
  "n_rows_test": 2265,
  "n_users_train": 216,
  "n_users_test": 216,
  "usuarios_compartidos_train_test": 216,
  "baseline_mediana_train": 60.96

## 2. Corte temporal, variables y predicciones

Construimos cada ejemplo con siete días consecutivos anteriores a la fecha que estimamos. Separamos los datos por fecha para evitar que información posterior aparezca en el entrenamiento. Las mismas personas pueden estar en ambos periodos; por eso, esta evaluación representa fechas futuras de usuarios ya observados y no permite medir el desempeño en usuarios nuevos.

In [3]:
OUT = ROOT / 'datos' / 'processed' / 'e3'
with (OUT / 'dataset_modelamiento.csv').open(encoding='utf-8-sig', newline='') as f: dataset = list(csv.DictReader(f))
with (OUT / 'predicciones_test.csv').open(encoding='utf-8-sig', newline='') as f: predicciones = list(csv.DictReader(f))
corte = metricas_e3['fecha_corte_temporal']
train_dates = [r['fecha_objetivo'] for r in dataset if r['particion']=='train']
test_dates = [r['fecha_objetivo'] for r in dataset if r['particion']=='test']
assert max(train_dates) == corte and min(test_dates) > corte
assert all(r['fecha_features_hasta'] < r['fecha_objetivo'] for r in dataset)
usuarios_train = {r['usuario_id'] for r in dataset if r['particion']=='train'}
usuarios_test = {r['usuario_id'] for r in dataset if r['particion']=='test'}
print('Ejemplos:',len(dataset),'| test:',len(predicciones),'| corte:',corte,'| usuarios compartidos esperados:',len(usuarios_train & usuarios_test))
display(predicciones[:5])

Ejemplos: 10710 | test: 2265 | corte: 2008-02-02 | usuarios compartidos esperados: 216
[{'usuario_id': '102058', 'fecha_features_hasta': '2008-02-02', 'fecha_objetivo': '2008-02-03', 'pasos_7d_promedio': '2970.5714285714284', 'minutos_sesion_7d_promedio': '0.0', 'comidas_7d': '5.0', 'porciones_7d': '15.0', 'calorias_7d': '6530.499999999999', 'sodio_7d': '14709.4', 'grasa_saturada_7d': '95.69999999999999', 'azucar_7d': '74.7', 'fibra_7d': '35.8', 'proteina_7d': '497.40000000000003', 'target_fc_reposo': '63.636', 'particion': 'test', 'prediccion_mediana': '60.966', 'prediccion_ols': '59.71758310003966', 'error_abs_ols': '3.9184168999603415'}, {'usuario_id': '102058', 'fecha_features_hasta': '2008-02-03', 'fecha_objetivo': '2008-02-04', 'pasos_7d_promedio': '4234.285714285715', 'minutos_sesion_7d_promedio': '7.714285714285714', 'comidas_7d': '4.0', 'porciones_7d': '11.0', 'calorias_7d': '5286.0', 'sodio_7d': '6581.599999999999', 'grasa_saturada_7d': '85.3', 'azucar_7d': '145.6000000000000

## 3. Interpretación y reproducibilidad

En este corte, el modelo de referencia basado en la mediana obtuvo mejores resultados que la regresión lineal. Los valores negativos de R² indican que, según esta métrica, ambos modelos predicen peor que una referencia basada en el promedio de los valores observados. Por ello, no recomendamos usar este predictor. El análisis tampoco permite conclusiones causales ni clínicas. La evaluación corresponde a fechas futuras de usuarios que ya aparecen en los datos.

Para poder reconstruir el análisis más adelante, conservamos las fuentes originales, el registro de archivos y tamaños, la versión del código, las dependencias fijadas en uv.lock, la fecha de corte y las métricas. El registro actual no guarda una huella digital del contenido de cada archivo; recomendamos agregarla antes de usar este flujo en producción.

## Coherencia con E1 y E2

Mantuvimos el universo descriptivo de 216 usuarios. De sus 11 864 días elegibles, 41 no tienen mediciones nocturnas válidas. Dejamos nulos esos valores diarios y no los incluimos como objetivos de entrenamiento. La evaluación temporal mide fechas posteriores para usuarios observados, no para personas nuevas.